# 🚀 Introduction: Building Your First AI Application

Welcome to this hands-on workshop! Over the next few hours, you'll build **Q-Scout Pro**, a complete AI-powered research assistant from scratch.

## 🎯 What You'll Learn

This workshop isn't about building the "best" research assistant—there are likely more sophisticated tools out there. Instead, it's designed to **demystify how Generative AI applications work** by giving you hands-on experience with each component in a typical AI pipeline.

## 🧩 The AI Application Stack

Modern AI apps combine multiple technologies:

- **Data Acquisition**: Fetching and processing information (APIs, PDFs, web scraping)
- **Language Models**: Using LLMs to understand and generate text
- **Speech Technologies**: Converting between text and audio (TTS/STT)
- **Search & Retrieval**: Finding relevant information from the web
- **User Interfaces**: Creating interactive experiences with Streamlit

## 💡 Learning Philosophy

**This is a learning sandbox**, not production code. You'll:
- Understand *why* each component exists
- See *how* they connect together
- Get *practical experience* integrating real APIs
- Build intuition for debugging AI pipelines

By the end, you won't just have a working app—you'll understand the building blocks well enough to create your own AI projects.

## ⚠️ What This Workshop Is NOT

- ❌ Not a course on optimal AI architecture
- ❌ Not production-ready code you should deploy as-is
- ❌ Not an exhaustive guide to every AI tool available
- ❌ Not a deep dive into AI theory or mathematical foundations
- ❌ Not intended to teach you everything about AI

## ✅ What This Workshop IS

- ✅ A practical introduction to common AI components
- ✅ A foundation for understanding how AI apps fit together
- ✅ A starting point for your own experimentation
- ✅ A way to build confidence working with APIs and models
- ✅ **A hands-on demonstration of how different pieces come together to make a final product**

##

Hopefully, by the end, you'll have a clear picture of how these various technologies integrate to create a functional AI application!

Let's get started! 🎓

# 🔌 What is an API?

**API** stands for **Application Programming Interface**. Think of it as a **waiter in a restaurant**:

- **You (the customer)** want food but can't go into the kitchen
- **The waiter** takes your order to the kitchen and brings back your food
- **The kitchen** prepares the food but doesn't interact with customers directly

### In Programming Terms

An **API** is a messenger that:
1. Takes your **request** (e.g., "get weather data")
2. Delivers it to a **server/system**
3. Returns the **response** back to you (e.g., "72°F, sunny")

### Why Use APIs?

- ✅ **Access data/services** you don't own (Google Maps, weather, payment systems)
- ✅ **Separation of concerns**: Frontend apps talk to backend servers via APIs
- ✅ **Reusability**: One API can serve mobile apps, websites, and scripts
- ✅ **Security**: APIs control what data users can access

### Real-World Example

When you check the weather on your phone:
- Your app sends an API request: `GET weather for New York`
- The weather service's API responds: `{"temp": 72, "condition": "sunny"}`
- Your app displays: ☀️ **72°F in New York**

### In This Workshop

The `api.py` file creates a **simple API service** with two endpoints:
- One that greets you by name
- One that echoes back whatever data you send

This teaches you how APIs **receive requests** and **send responses**—the foundation of all web services!


## 🌐 Our Own API Service

Let's run `api.py`

### Endpoint: `/api/greet` (GET)

- Takes a `name` from the URL query parameter (e.g., `?name=Bob`)
- If no name is provided, defaults to `"World"`
- Returns a JSON greeting: `{"message": "Hello, Bob!"}`

### Endpoint: `/api/echo` (POST)

- Accepts a JSON payload in the request body
- Returns the data unchanged
- Response format: `{"received": <your_original_data>}`

In [38]:
import requests
print(requests.get("http://127.0.0.1:5000/api/greet?name=Anshul").json())

{'message': 'Hello, Anshul!'}


# 🎓 Workshop: Building an AI Research Assistant (Q-Scout Pro)

Welcome! This notebook divides the development of a complex AI application into 4 digestible modules. By the end of these modules, you will have all the building blocks to create the `Q-Scout Pro` application.

### 🗺️ Roadmap
- **Module 0:** Environment Setup & Dependencies
- **Module 1:** Data Acquisition (ArXiv, Search, PDFs)
- **Module 2:** Intelligence & Media (LLMs, TTS)
- **Module 3:** Building the UI (Streamlit, State Management)

## 🛠️ Module 0: Setup & Foundations

Before we build, we need to install the necessary tools. The app relies on several powerful libraries:

*   **streamlit**: For the web interface.
*   **arxiv**: To search academic papers.
*   **pymupdf (fitz)**: To extract text from PDFs.
*   **groq**: For fast LLM inference.
*   **kokoro / soundfile**: For Text-to-Speech (TTS).
*   **tavily-python**: For AI-optimized web search.

In [39]:
# Install dependencies (Run this once)
%pip install streamlit arxiv pymupdf groq streamlit-mic-recorder soundfile numpy requests tavily-python
# Note: 'kokoro' installation might depend on your specific setup or local files.
# %pip install kokoro-onnx  # Example if using the ONNX version

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [40]:
import os
import io
import re
import time
import numpy as np
import requests
import arxiv
import fitz  # PyMuPDF
from groq import Groq

# Setup API Keys (Replace with your actual keys for testing in notebook)
# In the final app, these come from os.environ
##os.environ["GROQ_API_KEY"] = "gsk_..." 
##os.environ["TAVILY_API_KEY"] = "tvly-..."
## or run export commands in terminal:
# export GROQ_API_KEY="gsk_..."
# export TAVILY_API_KEY="tvly-..."

print("Libraries imported and Environment set!")

Libraries imported and Environment set!


## 📚 Module 1: Data Acquisition

An AI assistant is only as good as its data. In this module, we learn how to fetch academic papers and extract their content.

### What is ArXiv?

ArXiv is a free, open-access repository of electronic preprints (e-prints) for scientific papers in physics, mathematics, computer science, and related fields. Researchers upload their work before or alongside formal peer review, making cutting-edge research publicly available.

### The ArXiv API

ArXiv provides a public API that allows programmatic searching and retrieval of paper metadata (titles, authors, abstracts, PDF links, submission dates, etc.). The Python `arxiv` library wraps this API, making it easy to query and iterate over results.

### Main Ideas Behind the Code

1. **Date-Range Query**: The function calculates a UTC-aware cutoff date (`days` ago) and formats it to build a query string that filters papers by `submittedDate` within the desired window.

2. **Building the Search Query**: The query combines a keyword search (`all:"{query_text}"`) with a date range (`submittedDate:[start TO end]`) using ArXiv's query syntax.

3. **Client and Search Objects**: An `arxiv.Client()` handles network requests, while `arxiv.Search()` encapsulates the query, result limit, and sort order.

4. **Timezone-Aware Filtering**: Each result's `published` datetime is normalized to UTC. A secondary check ensures only papers after the cutoff are kept, guarding against API quirks.

5. **Result Collection**: Matching papers are returned as a list of tuples containing the title, PDF URL, and publication datetime.

In [58]:
from sickle import Sickle
from datetime import datetime, timedelta

def harvest_and_search_arxiv(keyword, days=7, arxiv_set='cs'):
    # 1. Calculate our absolute cutoff point
    cutoff_date = datetime.utcnow() - timedelta(days=days)
    from_date_str = cutoff_date.strftime('%Y-%m-%d')
    
    sickle = Sickle('http://export.arxiv.org/oai2')
    print(f"Harvesting records modified since {from_date_str}...")
    
    records = sickle.ListRecords(**{
        'metadataPrefix': 'oai_dc',
        'from': from_date_str,
        'set': arxiv_set
    })

    results = []
    
    try:
        for record in records:
            if not record.metadata:
                continue 
            
            # 2. Extract the actual paper dates from Dublin Core metadata
            paper_dates = record.metadata.get('date', [])
            is_truly_new = False
            
            # arXiv usually lists dates as YYYY-MM-DD. 
            # We check if *any* of the listed dates for this paper are within our timeframe
            for d_str in paper_dates:
                try:
                    # Grab just the first 10 characters (YYYY-MM-DD) to be safe
                    d_obj = datetime.strptime(d_str[:10], "%Y-%m-%d")
                    if d_obj >= cutoff_date:
                        is_truly_new = True
                        break
                except ValueError:
                    pass # Ignore poorly formatted dates
            
            # 3. If the paper's actual dates are older than our cutoff, skip it!
            if not is_truly_new:
                continue

            # Safely extract title and abstract
            title = record.metadata.get('title', [''])[0].replace('\n', ' ')
            abstract = record.metadata.get('description', [''])[0].replace('\n', ' ')
            
            # LOCAL SEARCH
            if keyword.lower() in title.lower() or keyword.lower() in abstract.lower():
                urls = [u for u in record.metadata.get('identifier', []) if 'arxiv.org/abs' in u]
                url = urls[0] if urls else "No URL found"
                # Grab the most recent date for display
                display_date = paper_dates[0] if paper_dates else "Unknown"
                
                results.append((title, url, display_date))
                
    except Exception as e:
        print(f"\nFinished harvesting or encountered an error: {e}")
        
    return results

# Test it
papers = harvest_and_search_arxiv("LLM", days=7, arxiv_set='cs')

print(f"\n--- Found {len(papers)} matching NEW papers ---")
for title, url, date in papers:
    print(f"Found ({date}): {title[:75]}... -> {url}")

Harvesting records modified since 2026-02-06...

--- Found 1066 matching NEW papers ---
Found (2024-06-05): Synthetic Oversampling: Theory and A Practical Approach Using LLMs to Addre... -> http://arxiv.org/abs/2406.03628
Found (2024-05-31): Generative AI voting: fair collective choice is resilient to LLM biases and... -> http://arxiv.org/abs/2406.11871
Found (2024-08-21): EAGLE: Elevating Geometric Reasoning through LLM-empowered Visual Instructi... -> http://arxiv.org/abs/2408.11397
Found (2024-09-17): Kahaani: A Multimodal Co-Creative Storytelling System... -> http://arxiv.org/abs/2409.11261
Found (2024-09-25): Building Multilingual Datasets for Predicting Mental Health Severity throug... -> http://arxiv.org/abs/2409.17397
Found (2024-10-17): AgentDrug: Utilizing Large Language Models in An Agentic Workflow for Zero-... -> http://arxiv.org/abs/2410.13147
Found (2024-10-26): RARe: Retrieval Augmented Retrieval with In-Context Examples... -> http://arxiv.org/abs/2410.20088
Found (2024

## PyMuPDF (fitz) — A simple, layman explanation

- PyMuPDF is a Python wrapper around a PDF-handling library. Think of it as a tool that lets your Python code open a PDF (or other document), look inside, and work with the pieces.

- Core idea: a document is made of pages. You open a file (or bytes) and then ask for a page by number. Each page is the unit you read, render, or modify.

- Text and structure: pages contain text, images, and drawing elements. PyMuPDF can extract text in different ways — plain text, grouped “blocks” or a structured dictionary that includes coordinates. That lets you get the words or know where on the page they appear.

- Images and rendering: you can render a page to an image (a Pixmap) at a chosen resolution. This is how thumbnails or raster screenshots are produced.

- Coordinates and transforms: pages use a coordinate system. You can scale, rotate, or crop using simple transform matrices so extracted images/text match the size or region you need.

- Annotations and editing: PyMuPDF can read and create annotations (highlights, links, notes), insert or remove pages, and save edited documents.

- Searching and selection: you can search text on a page, find exact matches, and get bounding boxes for the matches to highlight or extract surrounding content.

- Practical flow:
    1. Open document (file path or bytes).
    2. Pick a page.
    3. Extract text or render to an image, or modify page contents/annotations.
    4. Save if you made changes.

- Why it’s useful: it’s fast, lets you work with PDFs programmatically without needing a full PDF editor, and gives both simple text output and precise control when you need layout/position info.

In [59]:
# 2. Downloading & Extracting Text from PDF
def extract_text_from_url(pdf_url):
    # Convert abstract URL to PDF URL if needed
    # e.g. http://arxiv.org/abs/2406.03628 -> http://arxiv.org/pdf/2406.03628
    pdf_url = pdf_url.replace('/abs/', '/pdf/')
    
    response = requests.get(pdf_url)
    pdf_content = response.content
    
    # Open with Fitz (PyMuPDF)
    text = ""
    with fitz.open(stream=io.BytesIO(pdf_content), filetype="pdf") as doc:
        # In the app, we sometimes only read specific pages to save time
        # Here, let's just read the first page
        if len(doc) > 0:
            text += doc[0].get_text()
            
    return text

# Test extraction
if papers:
    test_url = papers[0][1]
    content = extract_text_from_url(test_url)
    print(f"--- Extracted Content (First Page) ---\n{content[:500]}...")

--- Extracted Content (First Page) ---
Theoretical Foundations of Synthetic Oversampling and
Augmentation: A Transformer-Based Approach for
Imbalanced Data
Ryumei Nakada1*
Yichen Xu2∗
Lexin Li3†
Linjun Zhang4†
February 10, 2026
Abstract
Imbalanced classification and spurious correlation are common challenges in data science
and machine learning. Both issues are linked to data imbalance, with certain groups of data
samples significantly underrepresented, which in turn would compromise the accuracy, robust-
ness, and generalizability o...


## 🧠 Module 2: Intelligence & Media

Now that we have text, we need to process it using an LLM (Groq) and optionally convert it to speech.

### What is an LLM?

**LLM** stands for **Large Language Model**. It's an AI system trained on massive amounts of text that can understand and generate human-like language. Think of it as a very advanced autocomplete—you give it text, and it predicts what should come next. Examples include ChatGPT, Claude, and Llama.
### Why Use an External API?

Running a large language model locally on your laptop is usually impractical for several reasons:

- **Size**: LLMs like Llama-70B have billions of parameters, requiring tens of gigabytes of RAM/VRAM.
- **Hardware**: Most laptops lack the powerful GPUs (like NVIDIA A100s) needed to run these models at usable speeds.
- **Speed**: Even if you could run a smaller model locally, inference would be slow—taking minutes instead of seconds.
- **Setup Complexity**: Installing and configuring local LLMs involves dependencies, model downloads, and technical troubleshooting.

### What Do We Sacrifice?

Using an external API comes with trade-offs:

- **Privacy**: Your prompts and data are sent to third-party servers. For sensitive research or proprietary information, this may be unacceptable.
- **Cost**: API calls are billed per token (words processed). Heavy usage can become expensive.
- **Latency**: Network round-trips add delay, especially with poor internet connections.
- **Dependency**: If the API goes down or changes its terms, your application breaks.
- **Rate Limits**: APIs often restrict how many requests you can make per minute/day.

### Why Explore Local LLMs?

Despite the challenges, running LLMs locally is becoming more feasible and may be worth exploring:

- **Full Privacy**: Your data never leaves your machine.
- **No Ongoing Costs**: After setup, there are no per-query charges.
- **Offline Access**: Works without internet connectivity.
- **Customization**: You can fine-tune models for your specific domain.
- **Emerging Tools**: Projects like `llama.cpp`, `Ollama`, and quantized models (e.g., GGUF format) now allow smaller models to run on consumer hardware with acceptable speed.

For prototyping and learning, APIs like Groq are ideal. For production systems handling sensitive data, local deployment deserves serious consideration.

By using an external API like Groq, you offload all the heavy computation to specialized servers. You just send text over the internet and get a response back in seconds, without needing expensive hardware or complex setup.
### What is Groq?

**Groq** is a company that provides an API (a way for your code to talk to their servers) to run LLMs very fast. Instead of running a huge AI model on your own computer, you send a request over the internet, Groq's servers process it, and they send back the AI's response.

### How Does an LLM Work?

At a high level, an LLM processes text as follows:

1. **Tokenization**: Your input text is broken into small pieces called "tokens" (roughly words or word fragments).
2. **Context Window**: The model considers all tokens together (up to a limit, e.g., 8,000 or 128,000 tokens depending on the model).
3. **Prediction**: Based on patterns learned during training, the model predicts the most likely next token, then the next, and so on—generating a coherent response.

### How LLMs Work Technically

Under the hood, LLMs use a **transformer architecture** with these key components:

- **Embeddings**: Each token is converted into a high-dimensional vector (e.g., 4096 dimensions) that captures semantic meaning.
- **Attention Mechanism**: The model computes relationships between all tokens simultaneously using "self-attention," allowing it to understand context regardless of distance in the text.
- **Layers**: Transformers stack dozens of layers (e.g., 70B models may have 80+ layers), each refining the representation.
- **Feed-Forward Networks**: After attention, each token passes through dense neural network layers that transform its representation.
- **Softmax Output**: The final layer produces a probability distribution over the entire vocabulary (~32,000-100,000 tokens), and the highest-probability token is selected.

**Key parameters:**
- **Temperature**: Controls randomness. Lower (0.0-0.3) = deterministic; higher (0.7-1.0) = creative.
- **Top-p (nucleus sampling)**: Only considers tokens within the top cumulative probability mass.
- **Max tokens**: Limits response length to control costs and prevent runaway generation.

### What are Embeddings?

**Embeddings** are numerical representations of text as vectors (lists of numbers) in high-dimensional space. Each word, sentence, or token is mapped to a point in this space where semantically similar items are positioned closer together.

**Key insight**: Words with similar meanings end up near each other in vector space, enabling mathematical operations on language.

### Vector Arithmetic with Words

Remarkably, embeddings support meaningful arithmetic:

| Operation | Result |
|-----------|--------|
| `king - man + woman` | ≈ `queen` |
| `paris - france + italy` | ≈ `rome` |
| `bigger - big + small` | ≈ `smaller` |

This works because the vector difference `king - man` captures the concept of "royalty without maleness." Adding `woman` reintroduces gender, landing near `queen`.

**Why this matters**: Embeddings let LLMs understand relationships, analogies, and context mathematically—transforming fuzzy human language into precise numerical operations that neural networks can process.

### What is a Prompt?

A **prompt** is the text you send to the LLM as input. It's your question, instruction, or the context you want the model to work with. The quality of your prompt directly affects the quality of the response.

**Example prompt**: `"Explain quantum computing in simple terms."`

### What is a System Prompt?

A **system prompt** (or system message) is special instruction given to the LLM that defines its behavior, personality, or role for the entire conversation. It's separate from the user's question and typically set once at the start.

**Example**: `"You are a helpful research assistant."` — This tells the LLM to behave like an expert assistant rather than a generic chatbot.

### Message Roles Explained

When using chat-based APIs, messages have **roles**:

| Role | Purpose |
|------|---------|
| `system` | Sets the AI's behavior/personality (invisible to users) |
| `user` | The human's input or question |
| `assistant` | The AI's previous responses (for multi-turn context) |

This structure allows the LLM to understand conversation history and maintain consistent behavior.

### Understanding the Code

```python
client_groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))
```
- Creates a "client" (a connection) to Groq's service using your secret API key (stored in environment variables for security).

```python
def ask_llm(prompt, model="llama-3.3-70b-versatile"):
```
- Defines a function that takes your question (`prompt`) and optionally a model name.

```python
messages=[
    {"role": "system", "content": "You are a helpful research assistant."},
    {"role": "user", "content": prompt}
]
```
- **System message**: Sets the AI's personality/behavior.
- **User message**: Your actual question or instruction.

```python
return completion.choices[0].message.content
```
- Extracts the AI's text response from the returned data.

### The Flow
1. You write a prompt (e.g., "Summarize this text...")
2. Your code sends it to Groq's servers
3. The LLM processes it and generates a response
4. You receive the answer back in your code

In [60]:
from dotenv import load_dotenv

load_dotenv()

True

In [61]:
import importlib
import os

# Debug: Check if the key is loaded
api_key = os.environ.get("GROQ_API_KEY")
print(f"API Key loaded: {api_key[:20]}..." if api_key else "API Key NOT found!")

client_groq = Groq(api_key=api_key)


def ask_llm(prompt, model="llama-3.3-70b-versatile"):
    try:
        completion = client_groq.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful research assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test the LLM
summary_prompt = f"Summarize this academic text in one sentence: {content[:1000]}"
summary = ask_llm(summary_prompt)
print(f"AI Summary: {summary}")

API Key loaded: gsk_zN8lVqnbUSGZYczX...
AI Summary: This article proposes a theoretical foundation for using transformer-based large language models to generate synthetic samples and augment imbalanced data, aiming to address common challenges in data science and machine learning such as imbalanced classification and spurious correlation.


In [62]:
# 2. Text Cleaning for Speech (Regex)
# Raw markdown often sounds bad in TTS (reading out asterisks, etc.)
def clean_for_speech(text):
    text = re.sub(r'\*+', '', text)           # Remove asterisks
    text = re.sub(r'#+\s', '', text)          # Remove headers
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text) # Clean links
    return text

clean_text = clean_for_speech(summary)
print(f"Cleaned for Speech: {clean_text}")

Cleaned for Speech: This article proposes a theoretical foundation for using transformer-based large language models to generate synthetic samples and augment imbalanced data, aiming to address common challenges in data science and machine learning such as imbalanced classification and spurious correlation.


## Text-to-Speech (TTS) with Kokoro (LOCAL, no internet needed)

In this section, we'll demonstrate how to use the **Kokoro** TTS engine to generate high-quality speech from text. 
This pipeline will later be integrated into the app to read out summaries.

### How Kokoro TTS Works

**Kokoro** is a text-to-speech (TTS) engine that converts written text into spoken audio. Here's a simple breakdown:

#### How TTS Models Like Kokoro Work Internally

Modern TTS systems like Kokoro typically use a **neural network architecture** with these components:

- **Text Encoder**: Converts input text into phoneme sequences and linguistic features
- **Acoustic Model**: Transforms linguistic features into mel-spectrograms (visual representations of audio frequencies over time)
- **Vocoder**: Converts mel-spectrograms into actual audio waveforms

The model learns from thousands of hours of recorded speech, capturing patterns like intonation, rhythm, and pronunciation. Different **voice embeddings** allow the same model to produce different speaker characteristics without retraining.

#### What Happens Step-by-Step

1. **Setup**: You create a `KPipeline` with a language code (`'a'` = American English). This loads the right pronunciation rules.

2. **Text → Sound Instructions**: Your text is broken into phonemes—the basic sounds of speech (like "cat" → /k/ /æ/ /t/).

3. **Sound Generation**: A neural network converts these phonemes into audio waveforms—the actual sound data.

4. **Voice Selection**: The `voice` parameter (e.g., `'af_jessica'`) changes the speaker's characteristics without retraining the model.

#### Understanding the Code

| Code | What It Does |
|------|--------------|
| `KPipeline(lang_code='a')` | Creates the TTS engine for American English |
| `pipeline(text, voice='af_jessica', speed=1.0)` | Generates audio from text with chosen voice and speed |
| `np.concatenate(audio_segments)` | Combines audio chunks into one continuous clip |
| `ipd.Audio(combined_audio, rate=24000)` | Plays the audio in the notebook (24kHz sample rate) |

#### Key Parameters

- **`voice`**: Which speaker voice to use
- **`speed`**: How fast to speak (1.0 = normal, 1.5 = faster)
- **`lang_code`**: Language/accent setting

In [63]:
import soundfile as sf
import numpy as np
import IPython.display as ipd
from kokoro import KPipeline

# Initialize the pipeline (lang_code='a' for American English)
# Ensure you have the model installed or downloaded as required by kokoro
pipeline = KPipeline(lang_code='a')

# Text to convert
text = "This is a demonstration of the Kokoro text-to-speech engine. It converts this text into natural-sounding speech."

print(f"Generating audio for: '{text}'")

# Generate audio
# 'af_bella' is a voice ID. Speed can be adjusted.
#generator = pipeline(text, voice='af_bella', speed=1.0)
## now use af_jessica
generator = pipeline(text, voice='af_jessica', speed=1.0)

# Collect audio segments
audio_segments = []
for _, _, audio in generator:
    audio_segments.append(audio)

# Combine and play
if audio_segments:
    combined_audio = np.concatenate(audio_segments)
    # 24000 is the default sample rate for Kokoro
    ipd.display(ipd.Audio(combined_audio, rate=24000))
else:
    print("No audio generated.")

/home/akapoor/anaconda3/envs/ml/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/akapoor/anaconda3/envs/ml/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Generating audio for: 'This is a demonstration of the Kokoro text-to-speech engine. It converts this text into natural-sounding speech.'


## Speech-to-Text (STT) with Groq Whisper

Now let's reverse the process. We will take the audio we just generated (or any audio file) and transcribe it back to text using the **Groq Whisper** model. 

In your Streamlit app, the audio bytes will come from the microphone widget (`mic_recorder`), but here we will simulate that by using the audio buffer we created in the previous step.

In [64]:
import io

# Use the recording.m4a file instead of the generated audio
audio_file_path = "recording.m4a"

print(f"Sending audio file '{audio_file_path}' to Groq...")

# 2. Call the Groq Whisper API
try:
    with open(audio_file_path, "rb") as audio_file:
        transcription = client_groq.audio.transcriptions.create(
            file=audio_file,
            model="whisper-large-v3",
            response_format="text"
        )
    print("\n📝 Transcription Result:")
    print(f"'{transcription}'")
except Exception as e:
    print(f"Error during transcription: {e}")


Sending audio file 'recording.m4a' to Groq...
Error during transcription: [Errno 2] No such file or directory: 'recording.m4a'


## 🔍 Web Search with Tavily API

### What is Tavily?

**Tavily** is an AI-optimized search engine designed specifically for LLM applications. Unlike traditional search engines that return links for humans to click, Tavily returns structured, content-rich results that are easy for AI systems to consume and process.

### Why Use Tavily Instead of Google?

| Feature | Traditional Search | Tavily |
|---------|-------------------|--------|
| **Output** | Links + snippets | Full extracted content |
| **Format** | HTML pages to parse | Clean JSON for LLMs |
| **Purpose** | Human browsing | AI/RAG pipelines |
| **Rate limits** | Strict, expensive | Developer-friendly |

### How the API Works

1. **Initialize Client**: Create a `TavilyClient` with your API key
2. **Send Query**: Call `tavily.search()` with your search terms
3. **Receive Results**: Get structured JSON with titles, URLs, and extracted content

### Understanding the Code

```python
tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
```
- Creates a connection to Tavily's service using your API key

```python
response = tavily.search(query=query, max_results=5)
```
- Sends your search query and limits results to 5

```python
response['results']  # List of search results
result.get('title')  # Page title
result.get('url')    # Source URL
result.get('content') # Extracted text content
```
- Each result contains pre-extracted content ready for LLM consumption

### Key Parameters

| Parameter | Purpose |
|-----------|---------|
| `query` | Your search terms |
| `max_results` | Number of results to return |
| `search_depth` | `"basic"` or `"advanced"` (deeper extraction) |
| `include_answer` | Get an AI-generated summary |

### Use Case in Q-Scout Pro

Tavily enables the research assistant to fetch real-time web information when ArXiv papers aren't sufficient—for example, finding documentation, news, or supplementary resources.

In [65]:
from tavily import TavilyClient

# Web Search with Tavily API
# Tavily is an AI-optimized search engine designed for LLM applications


# Initialize the Tavily client
tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

# Perform a web search
query = "IISER Pune website"
response = tavily.search(query=query, max_results=5)

print(f"Search query: '{query}'\n")
print("--- Search Results ---")
for i, result in enumerate(response['results'], 1):
    print(f"\n{i}. {result.get('title', 'No title')}")
    print(f"   URL: {result.get('url', 'N/A')}")
    print(f"   Content: {result.get('content', 'No content')[:300]}...")

Search query: 'IISER Pune website'

--- Search Results ---

1. IISER Pune (@IISERPune) / Posts / X - Twitter
   URL: https://x.com/IISERPune
   Content: IISERPune. Location: Pune, India. Website: http://www.iiserpune.ac.in. Joined: Mar 23, 2018. 4060Posts. 62Following. 31618Followers. ✓Verified. Official Page of...

2. Indian Institute of Science Education and Research (IISER), Pune
   URL: https://www.indiascienceandtechnology.gov.in/organisations/centres-of-higher-learning/iisers/indian-institute-science-education-and-research-iiser-pune
   Content: The Indian Institute of Science Education and Research (IISER), Pune is a premier institute dedicated to research and teaching in the basic sciences....

3. Indian Institute of Science Education and Research, Pune - Wikipedia
   URL: https://en.wikipedia.org/wiki/Indian_Institute_of_Science_Education_and_Research,_Pune
   Content: # Indian Institute of Science Education and Research, Pune. Indian Institute Science Education and Research, 

# 🎯 Getting Structured Output from LLMs

When working with LLMs, you often need responses in a specific format (JSON, lists, tables) rather than free-form text. Here's how to achieve structured outputs:

## Why Structured Output Matters

- ✅ **Parsing**: Easy to extract specific fields programmatically
- ✅ **Validation**: Can check if output matches expected schema
- ✅ **Integration**: Direct use in databases, APIs, or UI components
- ✅ **Reliability**: Reduces ambiguity in LLM responses

## Methods to Get Structured Output

### 1. **Prompt Engineering** (Simple)

Explicitly ask the LLM to format its response:

```python
prompt = """
Extract information from this text and return ONLY valid JSON:
{
    "title": "paper title",
    "authors": ["author1", "author2"],
    "year": 2024
}

Text: [your text here]
"""
```

**Pros**: Works with any LLM  
**Cons**: Not guaranteed to follow format, may add extra text

### 2. **JSON Mode** (Groq/OpenAI Feature)

Many modern APIs support a `response_format` parameter:

```python
completion = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}  # Forces JSON output
)
```

**Pros**: Guaranteed valid JSON  
**Cons**: Still need to define structure in prompt

### 3. **Function Calling / Tools** (Advanced)

Define a schema, and the LLM fills it:

```python
tools = [{
        "type": "function",
        "function": {
                "name": "extract_paper_info",
                "parameters": {
                        "type": "object",
                        "properties": {
                                "title": {"type": "string"},
                                "authors": {"type": "array", "items": {"type": "string"}},
                                "year": {"type": "integer"}
                        },
                        "required": ["title"]
                }
        }
}]
```

**Pros**: Strict validation, type safety  
**Cons**: More complex setup

### 4. **Regex + Parsing** (Fallback)

Extract JSON/data from free-form response:

```python
import json
import re

response = ask_llm(prompt)
# Find JSON block in response
json_match = re.search(r'\{.*\}', response, re.DOTALL)
if json_match:
        data = json.loads(json_match.group())
```

**Pros**: Works even if LLM adds extra text  
**Cons**: Brittle, may fail on malformed output

## Best Practices

1. **Be Explicit**: State format requirements clearly in prompt
2. **Provide Examples**: Show desired output structure
3. **Validate Output**: Always check response format before using
4. **Handle Errors**: LLMs may not follow instructions perfectly
5. **Use System Prompts**: Set role like "You are a data extraction assistant that only outputs valid JSON"

## Example: Extracting Paper Metadata

```python
def extract_paper_metadata(text):
        prompt = f"""
        Extract these fields from the academic paper text and return ONLY a JSON object:
        - title (string)
        - authors (array of strings)
        - publication_year (integer)
        - abstract (string, first 200 chars)
        
        Text: {text[:1000]}
        
        Return format:
        {{"title": "...", "authors": [...], "publication_year": 2024, "abstract": "..."}}
        """
        
        response = client_groq.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                        {"role": "system", "content": "You output only valid JSON, no extra text."},
                        {"role": "user", "content": prompt}
                ],
                response_format={"type": "json_object"}
        )
        
        return json.loads(response.choices[0].message.content)
```

## When to Use Each Method

| Use Case | Best Method |
|----------|-------------|
| Simple data extraction | Prompt engineering |
| Need guaranteed valid JSON | JSON mode |
| Complex multi-step workflows | Function calling |
| Working with legacy/unreliable models | Regex parsing |

By combining these techniques, you can make LLM outputs predictable and easy to integrate into larger systems!

In [66]:
    import json

    # Sample email to extract structured data from
    sample_email = """
    From: john.doe@techcorp.com
    To: hr@techcorp.com
    Subject: Vacation Request - June 2024

    Hi HR Team,

    I would like to request vacation leave from June 15th to June 25th, 2024 (10 business days).
    I will be traveling to Japan with my family.

    My contact number during this period will be +1-555-0123.
    In case of emergencies, please reach out to Sarah Johnson (sarah.j@techcorp.com) who will cover my responsibilities.

    Best regards,
    John Doe
    Senior Software Engineer
    Employee ID: EMP-2847
    """

    # Define extraction prompt with clear structure
    extraction_prompt = f"""
    Extract the following information from this email and return ONLY valid JSON with these exact fields:
    - from_email (string)
    - to_email (string)
    - employee_name (string)
    - employee_id (string)
    - position (string)
    - leave_start_date (string in YYYY-MM-DD format)
    - leave_end_date (string in YYYY-MM-DD format)
    - duration_days (integer)
    - reason (string)
    - emergency_contact_name (string)
    - emergency_contact_email (string)
    - phone_number (string)

    Email:
    {sample_email}

    Return only the JSON object, no additional text.
    """

    # Call Groq API with JSON mode
    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a data extraction assistant that outputs only valid JSON with no extra text or formatting."},
            {"role": "user", "content": extraction_prompt}
        ],
        response_format={"type": "json_object"}
    )

    # Parse and display the structured output
    extracted_data = json.loads(response.choices[0].message.content)

    print("📧 Extracted Email Data:\n")
    print(json.dumps(extracted_data, indent=2))

📧 Extracted Email Data:

{
  "from_email": "john.doe@techcorp.com",
  "to_email": "hr@techcorp.com",
  "employee_name": "John Doe",
  "employee_id": "EMP-2847",
  "position": "Senior Software Engineer",
  "leave_start_date": "2024-06-15",
  "leave_end_date": "2024-06-25",
  "duration_days": 10,
  "reason": "Vacation - traveling to Japan with family",
  "emergency_contact_name": "Sarah Johnson",
  "emergency_contact_email": "sarah.j@techcorp.com",
  "phone_number": "+1-555-0123"
}


In [67]:
import json

# Sample email from a student applicant
sample_email = """
From: alex.rivera@university.edu
To: careers@innovatech.com
Subject: Internship Application - Software Engineering Intern - Summer 2026

Dear Hiring Team,

I am writing to express my interest in the Software Engineering Internship for Summer 2026. 
I am currently a Junior at State University pursuing a B.S. in Computer Science with a 3.8 GPA.

I have experience with Python, React, and AWS from my previous project work and a 3-month stint 
at a local startup. I am available to start on May 20th, 2026, and can work through August 15th.

You can reach me at +1-444-555-0199 or find my portfolio at github.com/arivera-dev.

Thank you for your time and consideration.

Best regards,
Alex Rivera
"""

# Define extraction prompt with fields relevant to student recruitment
extraction_prompt = f"""
Extract the following information from this internship application email. 
Return ONLY valid JSON with these exact fields:
- applicant_name (string)
- applicant_email (string)
- university (string)
- degree_program (string)
- gpa (float or null)
- skills (array of strings)
- available_start_date (string in YYYY-MM-DD format)
- available_end_date (string in YYYY-MM-DD format)
- phone_number (string)
- portfolio_link (string or null)

Email:
{sample_email}

Return only the JSON object, no additional text.
"""

# Call Groq API with JSON mode
response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are a data extraction assistant that outputs only valid JSON."},
        {"role": "user", "content": extraction_prompt}
    ],
    response_format={"type": "json_object"}
)

# Parse and display the structured output
extracted_data = json.loads(response.choices[0].message.content)

print("🎓 Extracted Internship Application Data:\n")
print(json.dumps(extracted_data, indent=2))

🎓 Extracted Internship Application Data:

{
  "applicant_name": "Alex Rivera",
  "applicant_email": "alex.rivera@university.edu",
  "university": "State University",
  "degree_program": "B.S. in Computer Science",
  "gpa": 3.8,
  "skills": [
    "Python",
    "React",
    "AWS"
  ],
  "available_start_date": "2026-05-20",
  "available_end_date": "2026-08-15",
  "phone_number": "+1-444-555-0199",
  "portfolio_link": "github.com/arivera-dev"
}


# How Sentence Transformer Works

**Sentence Transformers** are models that convert sentences or paragraphs into dense vector representations (embeddings) that capture their semantic meaning. These embeddings allow you to compare, search, and cluster text based on meaning rather than just keywords.

## Key Concepts

- **Transformer Architecture**: Uses models like BERT, RoBERTa, or DistilBERT, which process text using self-attention to understand context and relationships between words.
- **Pooling**: After encoding each token, the model pools (averages or selects) the token embeddings to create a single vector for the entire sentence.
- **Semantic Similarity**: Sentences with similar meanings have embeddings that are close together in vector space (measured by cosine similarity).
- **Applications**: Useful for semantic search, FAQ retrieval, clustering, duplicate detection, and more.

## Typical Workflow

1. **Load Model**: Choose a pre-trained sentence transformer (e.g., `all-MiniLM-L6-v2`).
2. **Encode Text**: Convert sentences into embeddings using `model.encode()`.
3. **Compare**: Use cosine similarity to find sentences with similar meanings.
4. **Retrieve**: Find the most relevant documents or answers based on semantic closeness.

**Example:**  
If you encode "How do I apply for an internship?" and "What is the process for internship applications?", their embeddings will be very similar, even though the wording is different.

Sentence Transformers make it easy to build intelligent search and retrieval systems that understand meaning, not just keywords.

In [68]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Our "Database" of random Internship FAQs
faq_data = [
    "International Students: We sponsor CPT/OPT for summer interns.",
    "Hardware: Every intern receives a MacBook Pro and a 27-inch monitor.",
    "Housing: We do not provide housing, but we have a Slack channel for roommate finding.",
    "Mentorship: Each intern is paired with a Senior Engineer for weekly 1:1s.",
    "Interview Process: It consists of one coding signal and two 45-minute technical interviews.",
    "Salary: Engineering interns are paid $45/hour in most US locations."
]

model = SentenceTransformer('all-MiniLM-L6-v2')
faq_vectors = model.encode(faq_data)


In [69]:

# 2. The Student's Question
query = "Will I get a laptop or do I bring my own?"
query_vector = model.encode([query])

In [70]:
faq_vectors

array([[ 0.00611619,  0.01345273,  0.08280297, ..., -0.06956735,
        -0.01241146,  0.01683786],
       [ 0.01927376,  0.03886511,  0.0908149 , ..., -0.0180821 ,
        -0.0133848 ,  0.03589564],
       [ 0.03968575, -0.09331678,  0.03078783, ...,  0.05211119,
        -0.11725023,  0.01950661],
       [-0.04410776,  0.00539451,  0.04318426, ...,  0.03230814,
        -0.01748759, -0.02153629],
       [-0.06755342,  0.04136845,  0.00996015, ...,  0.10897615,
        -0.05448576, -0.03310341],
       [ 0.03823569, -0.00440435,  0.13592571, ..., -0.06959075,
         0.01956039, -0.04123868]], shape=(6, 384), dtype=float32)

In [54]:
query_vector

array([[-3.05569079e-02, -2.60456000e-02,  5.84887229e-02,
        -4.63161319e-02,  1.02927964e-02,  1.71591714e-02,
        -3.70661961e-03,  2.42348276e-02,  6.95647523e-02,
         3.80079187e-02,  7.49001801e-02, -2.12249327e-02,
        -1.59634196e-03, -8.69223475e-03,  1.00717671e-01,
        -1.73561685e-02, -2.65865936e-03, -1.40262857e-01,
        -2.84976661e-02, -1.87706295e-02, -5.66890985e-02,
         1.81233119e-02, -6.65541962e-02, -3.94506902e-02,
        -1.88379809e-02,  1.95975695e-02, -7.82831758e-03,
        -1.48053782e-03, -2.82217506e-02,  4.31643240e-02,
         2.64504179e-02, -3.20367850e-02, -3.59894671e-02,
         7.63355196e-02,  3.25528160e-02, -6.08687624e-02,
         8.53352994e-02, -4.91237491e-02,  3.35327312e-02,
        -3.77853625e-02, -4.81697097e-02, -8.09169486e-02,
         3.38885970e-02,  6.94072768e-02,  9.20513794e-02,
        -3.22612189e-02,  5.77564258e-03,  2.26489082e-02,
         9.63780656e-02,  1.38667636e-02,  8.75781814e-0

In [55]:
# Calculate similarities
scores = cosine_similarity(query_vector, faq_vectors)[0]
best_index = np.argmax(scores)
retrieved_context = faq_data[best_index]

print(f"🎯 Top Match: {retrieved_context}")
print(f"📊 Similarity Score: {scores[best_index]:.4f}")

🎯 Top Match: Hardware: Every intern receives a MacBook Pro and a 27-inch monitor.
📊 Similarity Score: 0.4634


In [56]:
# 1. Calculate similarities (Returns an array of scores)
scores = cosine_similarity(query_vector, faq_vectors)[0]

# 2. Get the indices of the Top 3 scores
# argsort returns indices to sort the array; [-3:] gets the last three (highest); [::-1] reverses it
top_3_indices = np.argsort(scores)[-3:][::-1]

# 3. Extract the text and scores
retrieved_chunks = [faq_data[i] for i in top_3_indices]
top_scores = [scores[i] for i in top_3_indices]

# Print the results for verification
print("🔍 Top 3 Relevant Contexts Found:")
for i, (text, score) in enumerate(zip(retrieved_chunks, top_scores), 1):
    print(f"{i}. [{score:.4f}] {text}")

# 4. Join them together for the LLM
context_for_llm = "\n".join(retrieved_chunks)

🔍 Top 3 Relevant Contexts Found:
1. [0.4634] Hardware: Every intern receives a MacBook Pro and a 27-inch monitor.
2. [0.1485] International Students: We sponsor CPT/OPT for summer interns.
3. [0.1149] Mentorship: Each intern is paired with a Senior Engineer for weekly 1:1s.


## Exercise: Try changing the query to
#### Will I get a laptop or do I bring my own? Also, what is the pay for interns?"
### See what contexts it retrieves!

In [57]:
import importlib
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Debug: Check if the key is loaded
api_key = os.environ.get("GROQ_API_KEY")
print(f"API Key loaded: {api_key[:20]}..." if api_key else "API Key NOT found!")

client_groq = Groq(api_key=api_key)

prompt = f"""Context: {context_for_llm} , Question: {query}"""

def ask_llm(prompt, model="llama-3.3-70b-versatile"):
    try:
        completion = client_groq.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "Answer the student's question based on the provided context. If the context does not contain the answer, say 'I don't know.'"},
                {"role": "user", "content": prompt}
            ]
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# Test the LLM
summary_prompt = prompt  # Using the same prompt for simplicity; in practice, you might want to rephrase it
summary = ask_llm(summary_prompt)
print(f"AI Summary: {summary}")

API Key loaded: gsk_zN8lVqnbUSGZYczX...
AI Summary: You will receive a laptop. Every intern is provided with a MacBook Pro.


## 🖥️ Module 3: Building the UI with Streamlit

Now we bring everything together into a user-friendly application.

### What is Streamlit?

**Streamlit** is a Python library that turns data scripts into shareable web apps in minutes. No front-end experience (HTML, CSS, JS) is required.

### Core Concepts

1.  **Data Flow**:
    Streamlit's architecture allows you to write apps like standard Python scripts. Whenever a user interacts with a widget, the entire script re-runs from top to bottom.

2.  **Displaying Data**:
    *   `st.write()`: The swiss-army knife command. It can print text, dataframes, charts, and more.
    *   `st.title()`, `st.header()`, `st.subheader()`: For structuring text.
    *   `st.markdown()`: Renders markdown text.

3.  **Widgets**:
    *   `st.button()`: Triggers an action.
    *   `st.text_input()`: Gets text from user.
    *   `st.sidebar`: Puts elements in a side panel.

4.  **Session State**:
    Since the script re-runs on every interaction, variables reset by default. `st.session_state` allows you to persist data (like chat history or search results) between re-runs.

### Why Streamlit for this Project?

It allows us to rapidly prototype a chat interface, handle audio input/output, and display complex paper summaries without writing a single line of HTML or dealing with HTTP requests manually.